# L23 · Uplift y targeting económico

**S23 · Lectura: Cap. 22 · 40–55 min**

## La pregunta

¿Qué acción produce un beneficio adicional y a qué coste?

## Qué harás

Evaluar uplift con varias acciones y llevarlo a una interfaz local.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Qini por acción y tres casos de la policy.

> **Idea clave:** Targeting causal es elegir una acción, no solo ordenar personas.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
rng = setup()
FEATURES = [f"x{i}" for i in range(8)]

In [ ]:
# Una policy elige acción y coste
import matplotlib.pyplot as plt
labels = ['acción A', 'acción B', 'uplift', 'coste']
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.plot(range(len(labels)), [0] * len(labels), "o-", lw=3)
for i, label in enumerate(labels):
    ax.text(i, 0.08, label, ha="center", va="bottom", fontsize=10)
ax.set_xlim(-.35, len(labels) - .65); ax.set_ylim(-.2, .3); ax.axis("off")
ax.set_title("Una policy elige acción y coste")
plt.tight_layout(); plt.show()

## 1. El mundo: tres brazos y dos efectos distintos (Listado 22.1)
Campaña **experimental**: cada cliente recibe control (40 %), SMS (30 %) o llamada (30 %) al azar. Los dos tratamientos tienen heterogeneidad **diferente**:
$$\tau_{SMS}(x)=0{,}06-0{,}05x_0+0{,}03x_1,\qquad \tau_{llamada}(x)=0{,}12+0{,}09x_1-0{,}02x_3.$$

In [ ]:
def make_churn_data(n=5000, seed=42):
    r = np.random.default_rng(seed); X = r.uniform(-1, 1, (n, 8))
    arm = r.choice(["control", "sms", "call"], size=n, p=[0.4, 0.3, 0.3])
    tau_sms = 0.06 - 0.05 * X[:, 0] + 0.03 * X[:, 1]; tau_call = 0.12 + 0.09 * X[:, 1] - 0.02 * X[:, 3]; base = 0.45 + 0.08 * X[:, 2]
    p = base + (arm == "sms") * tau_sms + (arm == "call") * tau_call
    Y = (r.uniform(size=n) < np.clip(p, 0, 1)).astype(int)
    df = pd.DataFrame(X, columns=FEATURES); df["y"], df["arm"] = Y, arm
    return df, {"sms": tau_sms, "call": tau_call}                      # la verdad va APARTE: solo para EVALUATION-CELL
df, VERDAD = make_churn_data(5000)
print(df["arm"].value_counts(normalize=True).round(2).to_dict(), "· retención media:", df["y"].mean().round(3))

## 2. Un estimador por acción contra el control (§22.4.1)
Cada $\tau_k$ se estima **solo con las filas de {control, acción k}** y se predice sobre **toda** la población. Comparar SMS contra llamada estimaría $\tau_{call}-\tau_{SMS}$, que no sirve para asignar (la alternativa relevante de cada acción es no hacer nada). Usamos un T-learner lineal (Ridge), suficiente para este DGP.

In [ ]:
def uplift_por_accion(df, accion, features=FEATURES):
    sub = df[df["arm"].isin(["control", accion])]; t = (sub["arm"] == accion).to_numpy(); X = sub[features].to_numpy(float); y = sub["y"].to_numpy(float)
    mk = lambda: RidgeCV(alphas=np.logspace(-2, 3, 8))
    Xall = df[features].to_numpy(float)
    return mk().fit(X[t], y[t]).predict(Xall) - mk().fit(X[~t], y[~t]).predict(Xall)
df["uplift_sms"] = uplift_por_accion(df, "sms"); df["uplift_call"] = uplift_por_accion(df, "call")
print("uplift medio estimado → SMS", df["uplift_sms"].mean().round(3), "· llamada", df["uplift_call"].mean().round(3))
# EVALUATION-CELL
print("uplift medio verdadero → SMS", VERDAD["sms"].mean().round(3), "· llamada", VERDAD["call"].mean().round(3))
print("correlación τ̂ con τ verdadero → SMS", np.corrcoef(df["uplift_sms"], VERDAD["sms"])[0, 1].round(3), "· llamada", np.corrcoef(df["uplift_call"], VERDAD["call"])[0, 1].round(3))
print("¿se puede derivar la llamada del SMS? corr(τ̂_SMS, τ_llamada verdadero) =", np.corrcoef(df["uplift_sms"], VERDAD["call"])[0, 1].round(3))

**Mira el resultado.** ¿Por qué no se puede obtener el uplift de la llamada «escalando» el del SMS?

**Respuesta:**

Porque cada acción tiene **su propio patrón de heterogeneidad** (el SMS depende de $x_0$ y $x_1$; la llamada de $x_1$ y $x_3$). La última cifra muestra que el uplift estimado del SMS casi no ordena bien el uplift verdadero de la llamada. Si el histórico no tiene variación en una acción (por ejemplo, solo hubo SMS), ese uplift **no existe**, por más columnas que se creen.



## 3. Evaluar el ranking: Qini por acción (§22.4.1)
Cada acción se evalúa sobre **su propia submuestra** (control vs esa acción). La asignación es aleatoria, así que el propensity es la proporción del brazo en esa submuestra.

In [ ]:
def qini_area(score, y, t, e):
    ystar = y * (t - e) / (e * (1 - e)); o = np.argsort(-score); cum = np.cumsum(ystar[o]) / len(y)
    frac = np.arange(1, len(y) + 1) / len(y); return float(np.mean(cum - frac * cum[-1])), cum
filas = {}
for acc in ("sms", "call"):
    sub = df[df["arm"].isin(["control", acc])]; t = (sub["arm"] == acc).to_numpy().astype(float); e = t.mean()
    q, _ = qini_area(sub[f"uplift_{acc}"].to_numpy(), sub["y"].to_numpy(float), t, e)
    azar = np.mean([qini_area(np.random.default_rng(s).normal(size=len(sub)), sub["y"].to_numpy(float), t, e)[0] for s in range(40)])
    filas[acc] = {"área de Qini del modelo": q, "área de Qini al azar (40 reps)": azar}
show(pd.DataFrame(filas).T.round(4))

## 4. Completa tú: asignación por net score con presupuesto (Alg. 10)
Con costos $c_k$ y presupuesto $B$, el **net score** de la acción $k$ para el cliente $x$ es $s_k(x)=\hat\tau_k(x)-\lambda\,c_k$, y se asigna la acción con mayor net score (o «nada», con score 0). $\lambda$ es el **costo sombra**: el precio implícito del presupuesto. El gasto $G(\lambda)$ **decrece** con $\lambda$, así que se busca por bisección el $\lambda$ que lo lleva a $B$.

`asignar_net_score(up, costos, budget)`, con `up` = `{"sms": array, "call": array}` y `costos` = `{"sms": 0.5, "call": 8.0}`, devuelve `(asignacion, lam, gasto)` con `asignacion` un array de textos (`"nada"`, `"sms"`, `"call"`).
Si el presupuesto **no restringe** (con $\lambda=0$ el gasto ya es $\le B$), devuelve la asignación con $\lambda=0$.

In [ ]:
# ✏️ Completa tú — asignar_net_score
# Pista: Para un lambda dado: S = columnas [0, up[a]-lam*costos[a] ...]; idx = S.argmax(1). Si gasto > budget, hay que SUBIR lambda.
def asignar_net_score(up, costos, budget, tol=1e-3, max_iter=80):
    """Devuelve (asignacion, lam, gasto). Bisección sobre lambda: el gasto decrece al subir lambda."""
    acciones = list(up); n = len(next(iter(up.values())))
    raise NotImplementedError("completa asignar_net_score")

In [ ]:
COSTOS = {"sms": 0.5, "call": 8.0}
up = {"sms": df["uplift_sms"].to_numpy(), "call": df["uplift_call"].to_numpy()}
_up = {"a": np.array([0.10, 0.02, 0.05, 0.01]), "b": np.array([0.30, 0.20, 0.04, 0.02])}; _c = {"a": 1.0, "b": 4.0}
def _prueba():
    asig, lam, g = asignar_net_score(_up, _c, budget=6.0)
    a0, l0, g0 = asignar_net_score(_up, _c, budget=1e6)
    return (g <= 6.0 + 1e-9, g > 0, l0 == 0.0, list(a0) == ["b", "b", "a", "b"])
lab_check("asignar_net_score", _prueba, [((), (True, True, True, True), 0)])

## 5. El presupuesto del libro no restringe (E5)
El Listado 22.2 usa `BUDGET = 100 000`. Con $n=5000$, gastar en llamadas a **todos** cuesta $5000\times8=40\,000$: el presupuesto nunca se alcanza y las tres estrategias acaban **tratando a todos** (con SMS, con llamada o con la mejor de las dos), sin que el presupuesto decida nada. Lo comprobamos y luego **bajamos el presupuesto** a 15 000.

In [ ]:
def uplift_verdadero(asig):
    u = np.where(asig == "sms", VERDAD["sms"], np.where(asig == "call", VERDAD["call"], 0.0)); return float(u.sum())
def uplift_estimado(asig):
    return float(np.where(asig == "sms", up["sms"], np.where(asig == "call", up["call"], 0.0)).sum())
def gasto_de(asig): return float(sum(COSTOS[a] * (asig == a).sum() for a in COSTOS))
def estrategias(budget):
    n = len(df); out = {}
    n_sms = min(int(budget / COSTOS["sms"]), n); a = np.full(n, "nada", dtype=object); a[np.argsort(-up["sms"])[:n_sms]] = "sms"; out["A (solo SMS)"] = a
    n_call = min(int(budget / COSTOS["call"]), n); b = np.full(n, "nada", dtype=object); b[np.argsort(-up["call"])[:n_call]] = "call"; out["B (solo llamadas)"] = b
    c, lam, _ = asignar_net_score(up, COSTOS, budget); out["C (net score, mixto)"] = c; out["_lam"] = lam
    return out
for budget in (100_000, 15_000):
    est = estrategias(budget); lam = est.pop("_lam")
    # EVALUATION-CELL: el uplift verdadero logrado
    show(pd.DataFrame({k: {"gasto": gasto_de(v), "uplift estimado": uplift_estimado(v), "uplift verdadero": uplift_verdadero(v), "% llamadas": 100 * (v == "call").mean()} for k, v in est.items()}).T.round(1)
         .rename_axis(f"presupuesto = {budget:,} (λ* = {lam:.4f})"))

**Mira el resultado.** Con 100 000 las tres estrategias gastan menos que el presupuesto. ¿Qué pasa con 15 000 y qué papel juega λ*?

**Respuesta:**

Con 100 000, el presupuesto no restringe: las tres estrategias tratan a todos y gastan 2 500, 40 000 y unos 37 700, muy por debajo de 100 000, con λ* = 0: el presupuesto no está decidiendo nada. Con 15 000 el gasto se satura y aparecen diferencias: C asigna SMS a casi todos (barato) y reserva las llamadas para quienes **justifican** los 7,5 de diferencia. λ* es el **precio sombra**: cuánto uplift se ganaría por cada unidad de presupuesto adicional.



## 6. Ranking frente a calibración (§22.5, ejercicio 22.6.3)
Multiplicamos `uplift_call` por 3 «porque la llamada parece más efectiva». El **ranking** (y por tanto el Qini) no cambia, pero el net score sí: la magnitud decide quién recibe la acción cara y cuánto se gasta.

In [ ]:
up3 = {"sms": up["sms"], "call": up["call"] * 3}
q1, _ = qini_area(df["uplift_call"].to_numpy()[df["arm"].isin(["control", "call"]).to_numpy()], df.loc[df["arm"].isin(["control", "call"]), "y"].to_numpy(float), (df.loc[df["arm"].isin(["control", "call"]), "arm"] == "call").to_numpy().astype(float), 0.3 / 0.7)
q3, _ = qini_area(3 * df["uplift_call"].to_numpy()[df["arm"].isin(["control", "call"]).to_numpy()], df.loc[df["arm"].isin(["control", "call"]), "y"].to_numpy(float), (df.loc[df["arm"].isin(["control", "call"]), "arm"] == "call").to_numpy().astype(float), 0.3 / 0.7)
print(f"Qini de uplift_call: {q1:.5f}  ·  Qini de 3×uplift_call: {q3:.5f}  (el ranking no cambió)")
def asignar_lambda(upd, costos, lam):
    n = len(next(iter(upd.values()))); acc = list(upd)
    S = np.column_stack([np.zeros(n)] + [upd[a] - lam * costos[a] for a in acc]); return np.array(["nada"] + acc, dtype=object)[S.argmax(1)]
asig_ref, lam_ref, _ = asignar_net_score(up, COSTOS, 15_000)
def fila(asig, upd):
    proy = float(np.where(asig == "sms", upd["sms"], np.where(asig == "call", upd["call"], 0.0)).sum())
    return {"% llamadas": 100 * (asig == "call").mean(), "gasto": gasto_de(asig), "uplift proyectado por el modelo": proy, "uplift verdadero logrado": uplift_verdadero(asig)}   # EVALUATION-CELL (última columna)
filas = {"uplift original, λ* resuelto": fila(asig_ref, up),
         "uplift de la llamada ×3, MISMO λ*": fila(asignar_lambda(up3, COSTOS, lam_ref), up3),
         "uplift de la llamada ×3, λ re-resuelto": fila(asignar_net_score(up3, COSTOS, 15_000)[0], up3)}
show(pd.DataFrame(filas).T.round(1))

**Mira el resultado.** El Qini no cambió, pero la asignación sí. ¿Qué enseñanza deja sobre auditar un modelo de uplift solo por su Qini o AUUC?

**Respuesta:**

Que un buen **ranking** no implica buena **calibración**. Si la regla se despliega con el mismo costo sombra $\lambda^*$ pero los uplifts están inflados, el **gasto real se dispara** y el uplift *proyectado* promete mucho más de lo que se entrega. Aun si se re-resuelve $\lambda$ para respetar el presupuesto, la composición casi no cambia, pero el uplift proyectado sigue sobreestimado: los números que se le dan a quien decide son incorrectos. Hay que comprobar la magnitud contra una referencia (p. ej., diferencia de medias por deciles en el experimento) antes de usar el net score. Multiplicar uplifts «a ojo» es el error que el capítulo señala en su tabla de riesgos.



## 7. Sensibilidad a los costos (±20 %)
Un costo mal medido cambia la política. Variamos el costo de la llamada y miramos la mezcla.

In [ ]:
filas = {}
for f in (0.8, 1.0, 1.2):
    cc = {"sms": COSTOS["sms"], "call": COSTOS["call"] * f}; asig, lam, g = asignar_net_score(up, cc, 15_000)
    filas[f"costo llamada ×{f}"] = {"% llamadas": 100 * (asig == "call").mean(), "% SMS": 100 * (asig == "sms").mean(), "λ*": lam}
show(pd.DataFrame(filas).T.round(3))

## 8. ¿«El 20 % superior captura ≈ 60 % del uplift»? (apéndice del plan)
El capítulo afirma esta cifra para su Figura 22.1. La medimos con el DGP del Listado 22.1: la fracción del uplift verdadero total que captura el 20 % superior según el uplift **estimado**, y el techo que alcanzaría un oráculo.

In [ ]:
# EVALUATION-CELL
for acc in ("sms", "call"):
    tv = VERDAD[acc]; k = int(0.2 * len(tv))
    est = tv[np.argsort(-df[f"uplift_{acc}"].to_numpy())[:k]].sum() / tv.sum(); orac = np.sort(tv)[::-1][:k].sum() / tv.sum()
    print(f"{acc:5s}: el 20 % superior por uplift estimado captura {100 * est:.0f} % del uplift total · techo (oráculo): {100 * orac:.0f} % · azar: 20 %")

**Mira el resultado.** ¿Se reproduce el 60 % del libro con este DGP?

**Respuesta:**

No: aun con el ordenamiento perfecto, el techo del oráculo es bastante menor que 60 %, porque la heterogeneidad del DGP es pequeña respecto al efecto medio (L22 §6: el cociente depende de $s/\mu$). La cifra de la Figura 22.1 pertenece a otro DGP y **no es una constante del método**. Es una fe de erratas candidata: se debe decir de qué DGP sale.



## 9. Supuesto y bitácora

In [ ]:
asig_f, lam_f, g_f = asignar_net_score(up, COSTOS, 15_000)
fila_b = bitacora(
    pregunta="¿Qué acción (nada, SMS, llamada) asignar a cada cliente con un presupuesto de 15 000?",
    metodo="T-learner lineal por acción contra el control; Qini por acción; net score con λ por bisección",
    estimando="Uplift τ_k(x) por acción y asignación óptima bajo presupuesto",
    numero_ic=f"gasto {g_f:.0f} ≤ 15000; λ*={lam_f:.4f}; {100 * (asig_f == 'call').mean():.0f}% llamadas, {100 * (asig_f == 'sms').mean():.0f}% SMS",
    supuesto=supuesto("observable", "medido", f"el presupuesto se cumple (gasto {g_f:.0f}) y las decisiones son binarias por acción; asignación aleatoria de los tres brazos"),
    decision="continuar", lab="L23")

---
## Qué queda para el libro
Quedan en el libro: `causalml`/`xgboost` (no instalados; aquí un T-learner lineal), el bootstrap de la curva Qini (caja 12.1) y el análisis de equidad de los tratados. El puente opcional con las pruebas de humo de B3 depende del repositorio y no se incluye.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*